# Example 13 — Download weather data and simulate millet in Niger

This notebook downloads daily TAMSAT rainfall and AgERA5 v2 climate data, prepares an `xarray.Dataset` on the rainfall grid, adds the repository's soil maps, and runs SARRA-Py through the same API as Example 1.

The data providers deliver **NetCDF files**. Example 1 instead reads an archived collection of daily GeoTIFFs. Both routes produce the same model input variables (`rain`, `tpMoy`, `ET0`, `rg`), but their source datasets and ET0 methods differ, so the two examples should not be treated as numerically identical.

TAMSAT rainfall is publicly downloadable. AgERA5 downloads require a Copernicus Climate Data Store account, acceptance of the dataset terms, and a configured [CDS API key](https://cds.climate.copernicus.eu/how-to-api). Install the project requirements before starting.


## 1. Locate the repository and import the API

This notebook works when Jupyter starts in the repository root or in `notebooks/`. Downloads are stored under `data/exemple_data/exemple_13/`; repeated runs reuse completed files.


In [ ]:
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import display

from sarra_py import (
    SimulationConfig,
    download_agera5_v2,
    download_tamsat_daily,
    load_model_parameters,
    override_parameters,
    prepare_remote_inputs,
    run_simulation,
    validate_input_dataset,
)

current_dir = Path.cwd().resolve()
if (current_dir / "data" / "assets").is_dir():
    project_root = current_dir
elif current_dir.name == "notebooks" and (current_dir.parent / "data" / "assets").is_dir():
    project_root = current_dir.parent
else:
    raise RuntimeError("Open this notebook from the repository root or notebooks/.")

cache_dir = project_root / "data" / "exemple_data" / "exemple_13"
assets_dir = project_root / "data" / "assets"
print(f"Repository: {project_root}")
print(f"Download cache: {cache_dir}")


## 2. Choose dates and a small area

The water balance starts on 1 April, **30 days before the earliest eligible sowing date**. This gives the soil water state time to respond to rainfall before the sowing decision. The box is `(north, west, south, east)` in degrees; keep it small for a first run. A full 250-day season requests 250 TAMSAT files and three AgERA5 monthly products per month, so the first download can take time.


In [ ]:
date_start = date(2017, 4, 1)
duration = 250
sowing_date = date(2017, 5, 1)
bounds = (13.9, 2.0, 13.7, 2.2)  # North, west, south, east (southern Niger).
date_end = date_start + timedelta(days=duration - 1)

print(f"Weather and simulation: {date_start} to {date_end}")
print(f"Earliest sowing: {sowing_date} ({(sowing_date - date_start).days} warm-up days)")
print(f"Area: {bounds}")


## 3. Download the daily forcing

TAMSAT v3.1 provides one daily NetCDF file with the filled rainfall field `rfe_filled`. The CDS API returns AgERA5 v2 data by variable and month; the downloader requests only the selected dates and area and caches the returned NetCDF files. Interrupted downloads can be rerun: completed cache entries are reused.

AgERA5 v2 supplies FAO56 Penman–Monteith reference evapotranspiration. Example 1's local raster archive uses Hargreaves ET0. This changes the forcing and can change simulated outcomes.


In [ ]:
# The first call retrieves one public rainfall file per day.
tamsat_files = download_tamsat_daily(
    date_start, duration, cache_dir / "tamsat", progress=True,
)

# CDS credentials and accepted terms are needed for this call.
agera5_files = download_agera5_v2(
    date_start, duration, bounds, cache_dir / "agera5", progress=True,
)
print(f"TAMSAT days: {len(tamsat_files)}")
print({name: len(files) for name, files in agera5_files.items()})


## 4. Prepare one model input dataset

TAMSAT rainfall defines the calculation grid. Its cell centres are about 0.0375° apart, while the native AgERA5 cells are about 0.1° apart. AgERA5 is downloaded with a 0.2° buffer (about two native cells) around the selected box so that edge TAMSAT cells have climate data. This uses a new cache entry if you previously downloaded the tighter box.

`prepare_remote_inputs` crops the sources and uses `rio.reproject_match` with nearest-neighbour resampling to place each AgERA5 field on TAMSAT coordinates. The soil rasters are aligned to those coordinates as well. Thus `base_data["tpMoy"]` and `base_data["rain"]` have the same array shape **after** alignment; the source AgERA5 file has its own coarser grid. Reprojection repeats climate values across several TAMSAT cells and adds no finer climate information. Preparation stops with an error if valid rainfall cells lack climate coverage.

The next cell prints native and model grid dimensions and spacings. It also converts temperature from kelvin to Celsius and radiation from J/m²/day to MJ/m²/day. The prepared weather variables have daily `(time, y, x)` dimensions; soil fields have `(y, x)`.


In [ ]:
base_data = prepare_remote_inputs(
    tamsat_files, agera5_files, bounds, assets_dir, date_start, duration,
)

# Compare source spacings with the actual calculation grid.
with xr.open_dataset(tamsat_files[0]) as tamsat_source:
    tamsat_step = abs(float(tamsat_source["lon"].diff("lon").median()))
with xr.open_dataset(agera5_files["tpMoy"][0]) as agera5_source:
    agera5_step = abs(float(agera5_source["lon"].diff("lon").median()))
    agera5_native_shape = (agera5_source.sizes["lat"], agera5_source.sizes["lon"])
model_step = abs(float(base_data.x.diff("x").median()))
assert abs(model_step - tamsat_step) < 1e-8
print(f"Longitude spacing: TAMSAT {tamsat_step:.4f}°, AgERA5 {agera5_step:.4f}°, model {model_step:.4f}°")
print(f"Downloaded AgERA5 tile: {agera5_native_shape[0]} × {agera5_native_shape[1]} native cells")
print(f"Model grid (TAMSAT): {base_data.sizes['y']} × {base_data.sizes['x']} pixels")
print(f"Dates: {base_data.time.values[0]} to {base_data.time.values[-1]}")
display(base_data)


## 5. Load parameters

The same variety, management and soil YAML files as Example 1 define the model settings. `DateSemis` is the earliest eligible sowing day; the actual simulated sowing date can depend on the water balance.


In [ ]:
# Load the three YAML files through the public parameter API.
# Explicit paths make it clear which values are used for this run.
base_parameters = load_model_parameters(
    project_root / "data" / "params" / "variety" / "millet_variety.yaml",
    project_root / "data" / "params" / "itk" / "millet_niger_2017.yaml",
    project_root / "data" / "params" / "soil" / "USA_iowa_V42.yaml",
)

# Replace only DateSemis in an independent parameter copy.
parameters = override_parameters(
    base_parameters, management={"DateSemis": sowing_date},
)

# Use the default NumPy engine and show progress across simulation days.
config = SimulationConfig(date_start=date_start, duration=duration, progress=True)
validate_input_dataset(base_data, config)
print(f"Sowing date used (DateSemis): {parameters.management['DateSemis']}")
print(f"Engine: {config.engine}; model: {config.model}")

## 6. Run the simulation

`run_simulation` starts from an independent copy of the prepared input dataset. The progress bar counts simulated days. Because the weather products differ from Example 1, compare this run on its own terms rather than expecting matching numerical outputs.


In [ ]:
# This is the shared entry point used by both notebooks and the CLI.
print(f"Running {duration} daily steps on a {base_data.sizes['y']} × {base_data.sizes['x']} grid...", flush=True)
result = run_simulation(base_data, parameters, config)
print("Simulation complete. Selected outputs:")

# Inspect time, spatial dimensions, and the two outputs used below.
result[["numPhase", "rdt"]]

## 7. Inspect development and final yield

Phase maps show how the simulated crop develops across the selected area. The yield map is a **model result**, not an observation. Cells without soil coverage are masked.


In [ ]:
# Compare the start, roughly one-third and two-thirds of the season,
# and the final day. The set removes duplicates for very short runs.
snapshot_days = sorted({0, duration // 3, 2 * duration // 3, duration - 1})
fig, axes = plt.subplots(1, len(snapshot_days), figsize=(16, 4), constrained_layout=True)

for ax, day_index in zip(axes, snapshot_days):
    # Hide cells without soil data before displaying the phenological phase.
    phase = result["numPhase"].isel(time=day_index).where(result["ru"].notnull())
    phase.plot(ax=ax, vmin=0, vmax=7, cmap="viridis", add_colorbar=False)

    # Use calendar dates so the maps are easier to relate to the season.
    day = date_start + timedelta(days=day_index)
    ax.set_title(day.isoformat())
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

fig.suptitle("Phenological phase progression")
plt.show()

In [ ]:
# Select the final simulated day and mask cells without soil information.
# rdt is reported in kg/ha; the colorbar carries that unit.
yield_map = result["rdt"].isel(time=-1).where(result["ru"].notnull())
fig, ax = plt.subplots(figsize=(8, 6))
yield_map.plot(ax=ax, cmap="YlGn", cbar_kwargs={"label": "kg/ha"})
ax.set_title(f"Simulated yield on {date_end.isoformat()}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

### Reuse the downloaded inputs

The cache can be reused for another parameter scenario or a rerun. For a separate CLI run, save the prepared dataset with `base_data.to_netcdf(path)` and point a `prepared_netcdf` configuration at that file. The CLI and notebook use the same `run_simulation` API.

Provider documentation: [TAMSAT daily rainfall](https://research.reading.ac.uk/tamsat/data-access/) and [AgERA5 v2](https://cds.climate.copernicus.eu/datasets/sis-agrometeorological-indicators).
